# Embedding comparison

All retrieval lives or dies on the embedder. This notebook benchmarks **three configurations** of the deterministic hashing embedder (varying dims, n-gram, and seed) against the golden Q&A set, measuring `recall@1` and `recall@3`.

> The configurations stand in for real providers (OpenAI text-embedding-3-small, Cohere embed-english-v3, Voyage voyage-3, …) so the notebook runs deterministically and offline. The control flow is identical to the real-provider case — swap `hash_embed(...)` for `shared.llm.embed(model=..., inputs=..., namespace=...)` and the rest works unchanged.

In [1]:
# %% Notebook bootstrap — never touches API keys directly.
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# CI / offline mode: replay cached responses instead of calling out.
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
import numpy as np
from shared.embedders import cosine_topk, hash_embed
from shared.loaders import load_corpus, load_golden_qa

DOCS = load_corpus()
QA = [q for q in load_golden_qa() if q.source_ids]  # exclude unanswerable
doc_texts = [d.title + '. ' + d.abstract for d in DOCS]
doc_ids = [d.arxiv_id for d in DOCS]
print('docs:', len(DOCS), '  answerable Qs:', len(QA))

docs: 10   answerable Qs: 26


## Configurations under test
Three settings, intentionally varied, to mimic the differences between real providers:

* **tiny-128** — 128 dims, unigram only. Stand-in for a small/cheap embedder.
* **balanced-256** — 256 dims, uni+bigram. Stand-in for a mid-tier embedder.
* **large-512** — 512 dims, uni+bigram. Stand-in for a frontier embedder.

In [3]:
CONFIGS = {
    'tiny-128':     dict(dims=128, ngram=(1, 1), seed=11),
    'balanced-256': dict(dims=256, ngram=(1, 2), seed=22),
    'large-512':    dict(dims=512, ngram=(1, 2), seed=33),
}

def recall_at_k(doc_vecs: np.ndarray, q_vecs: np.ndarray, ks=(1, 3, 5)) -> dict[int, float]:
    out: dict[int, float] = {}
    for k in ks:
        hits = 0
        for i, item in enumerate(QA):
            idx, _ = cosine_topk(q_vecs[i], doc_vecs, k=k)
            retrieved = {doc_ids[j] for j in idx}
            if set(item.source_ids) & retrieved:
                hits += 1
        out[k] = round(hits / len(QA), 4)
    return out

In [4]:
results = {}
for name, kwargs in CONFIGS.items():
    doc_vecs = hash_embed(doc_texts, **kwargs)
    q_vecs = hash_embed([q.question for q in QA], **kwargs)
    results[name] = recall_at_k(doc_vecs, q_vecs)
import json
print(json.dumps(results, indent=2))

{
  "tiny-128": {
    "1": 0.6923,
    "3": 0.9231,
    "5": 1.0
  },
  "balanced-256": {
    "1": 0.6923,
    "3": 0.9615,
    "5": 1.0
  },
  "large-512": {
    "1": 0.8077,
    "3": 0.9231,
    "5": 1.0
  }
}


## Reading the results
In the real-provider case you'd expect monotone improvements with model size — and you'd also see embedders with very different *failure modes* (some over-emphasize lexical overlap, others surface paraphrases). The committed `eval-snapshot.json` records these numbers so subsequent improvements (e.g. moving the comparison onto real providers) are reviewable in git.

## Switching to real providers
```python
from shared.llm import embed
doc_vecs = np.array(embed(model='openai/text-embedding-3-small', inputs=doc_texts, namespace=NS))
q_vecs   = np.array(embed(model='openai/text-embedding-3-small', inputs=[q.question for q in QA], namespace=NS))
```
Set `OPENAI_API_KEY` / `COHERE_API_KEY` / `VOYAGE_API_KEY` and rerun. The cached vectors land in `.llm-cache/01-rag/02-embedding-comparison.jsonl`, which you can commit to make future runs reproducible without API keys.